## Evaluate Results on ResOPS Hyperparameter Tuning
For the randomly selected and tuned reservoirs, compare optimal hyperparameters and % difference in validation error with Shasta baseline.

In [1]:
import os
print(f"Current Directory: {os.getcwd()}")
os.chdir("..")
print(f"New Directory: {os.getcwd()}")

Current Directory: c:\Users\mattc\Documents\DL-reservoir-modeling\additional_experiments
New Directory: c:\Users\mattc\Documents\DL-reservoir-modeling


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import torch

from src.data.data_processing import *
from src.data.data_fetching import *
from src.models.predict_model import flatten_rm_pad

### Helpers to convert MSE (val error) -> $R^2$

In [3]:
def processed_variance(res_id, transform_type='standardize', train_frac=0.6, val_frac=0.2, test_frac=0.2):
    '''
    Get the variance of processed outflow data for a given reservoir to help convert MSE -> R^2.
    Use biased variance to match MSE denominator (n vs n-1)
    '''
    # Mimic the training data-processing pipeline.
    df = resops_fetch_data(res_id=res_id, vars=['inflow', 'outflow', 'storage'])
    df['doy'] = df.index.to_series().dt.dayofyear
    df = df[f"{get_left_years([res_id])[res_id]}-01-01":'2020-12-31'].copy()

    pipeline = processing_pipeline(
        train_frac=train_frac,
        val_frac=val_frac,
        test_frac=test_frac,
        chunk_size=3*365,
        pad_value=-1,
        transform_type=transform_type,
        fill_na_method='mean'
    )
    ts_train, ts_val, ts_test = pipeline.process_data(df)

    # Extract outflow target tensors and remove padding using the shared helper.
    y_train = ts_train[:, :, [1]]
    y_val = ts_val[:, :, [1]]
    y_test = ts_test[:, :, [1]]

    _, y_train = flatten_rm_pad(y_hat=torch.zeros_like(y_train), y=y_train)
    _, y_val = flatten_rm_pad(y_hat=torch.zeros_like(y_val), y=y_val)
    _, y_test = flatten_rm_pad(y_hat=torch.zeros_like(y_test), y=y_test)

    return {
        'train': torch.var(y_train, unbiased=False).item(),
        'val': torch.var(y_val, unbiased=False).item(),
        'test': torch.var(y_test, unbiased=False).item()
    }

In [4]:
def mse_to_r2(mse, variance):
    '''
    Convert mean squared error to R^2 using R^2 = 1 - MSE / Var(y). Variance should be computed with n instead of n-1 to match MSE denominator
    '''
    if variance == 0:
        raise ValueError('variance must be non-zero to compute R^2')
    return 1 - mse / variance

In [5]:
df = pd.DataFrame(index=np.arange(10), columns=['reservoir ID', 'baseline_val_r2', 'best_val_r2', 'Δ validation r2', 'optimal hyperparameters'])

log_dir = Path("report/results/additional_experiments/hyperparameter_tuning_resops/logs")
for i, csv_file in enumerate(log_dir.glob("*_grid_search.csv")):
    # Read hyperparameter tuning log csv
    df_i = (pd.read_csv(csv_file, index_col=0)
        .groupby(['num_layers', 'hidden1', 'hidden2', 'dropout'], as_index=False)
        .mean(numeric_only=True)
        .drop(columns=['random_seed'], errors='ignore')
        .sort_values(by='val_error', axis=0))
    
    res_id = int(csv_file.stem.removesuffix("_grid_search"))
    res_val_variance = processed_variance(res_id)['val']
    baseline_val_error = df_i[
        (df_i['num_layers'] == 1) &
        (df_i['hidden1'] == 30) &
        (df_i['hidden2'] == 15) &
        (df_i['dropout'] == 0.3)
    ]['val_error'].values[0]
    df.loc[i, 'reservoir ID'] = res_id
    df.loc[i, 'baseline_val_r2'] = round(mse_to_r2(baseline_val_error, res_val_variance), 3)
    df.loc[i, 'best_val_r2'] = round(mse_to_r2(df_i['val_error'].min(), res_val_variance), 3)
    df.loc[i, 'Δ validation r2'] = round((df.loc[i, 'best_val_r2'] - df.loc[i, 'baseline_val_r2']), 3)
    df.loc[i, 'optimal hyperparameters'] = f'({df_i.iloc[0]["num_layers"]}, {df_i.iloc[0]["hidden1"]}, {df_i.iloc[0]["hidden2"]}, {df_i.iloc[0]["dropout"]})'

df

,reservoir ID,baseline_val_r2,best_val_r2,Δ validation r2,optimal hyperparameters
0,1026,0.6,0.628,0.028,"(2.0, 40.0, 40.0, 0.3)"
1,1067,0.322,0.387,0.065,"(2.0, 25.0, 35.0, 0.5)"
2,1112,0.533,0.567,0.034,"(1.0, 40.0, 50.0, 0.5)"
3,1645,0.598,0.634,0.036,"(2.0, 15.0, 15.0, 0.5)"
4,1744,0.689,0.723,0.034,"(2.0, 40.0, 45.0, 0.3)"
5,1756,0.55,0.562,0.012,"(1.0, 50.0, 40.0, 0.3)"
6,372,0.895,0.913,0.018,"(1.0, 45.0, 50.0, 0.3)"
7,423,0.799,0.842,0.043,"(2.0, 5.0, 15.0, 0.3)"
8,575,0.68,0.707,0.027,"(1.0, 10.0, 40.0, 0.5)"
9,601,0.325,0.349,0.024,"(1.0, 15.0, 50.0, 0.5)"
